# Modelo: XGBoost (Gradient Boosting)
**Responsable:** Ivan Angeles - Tree Team

## Introducción
En este notebook entrenaremos el modelo **XGBoost**. Dado que el equipo desarrolló diferentes opciones de limpieza (A y B) para ciertas variables en el *Feature Engineering*, implementaremos un ciclo iterativo ("Torneo") que probará automáticamente las **64 combinaciones posibles**. 

El objetivo es utilizar el 100% de las variables disponibles (incluyendo `City` y `Bank` codificadas) para que el algoritmo determine su verdadera importancia. Posteriormente, aplicaremos `GridSearchCV` sobre el dataset ganador para evitar sobreajuste y encontrar los hiperparámetros óptimos.

In [1]:
# ==========================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y ENTORNO
# ==========================================
import itertools
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("⏳ Cargando entorno de Feature Engineering...")
# Heredamos todas las funciones creadas por el equipo en src/preprocessing/
%run "feature-enginnering.ipynb"

# ==========================================
# 2. CONFIGURACIÓN DE COMBINACIONES A/B
# ==========================================
opciones = {
    'newExists': ['A', 'B'],
    'createJob': ['A', 'B'],
    'retainedJob': ['A', 'B'],
    'approvalDate': ['A', 'B'],
    'approvalFY': ['A', 'B'],
    'disbursementGross': ['A', 'B']
}

keys = list(opciones.keys())
combinaciones = list(itertools.product(*(opciones[k] for k in keys)))

mejor_f1 = 0
mejor_combo = None
mejor_df = None

# Cargamos el dataset crudo original
df_raw = pd.read_csv(project_root / "data" / "train.csv")

print("⏳ Iniciando torneo de 64 combinaciones A/B (esto tomará unos minutos)...")

# ==========================================
# 3. CICLO DE EVALUACIÓN (EL TORNEO)
# ==========================================
for combo in combinaciones:
    df_temp = df_raw.copy()
    
    # --- RECETA ESTRICTA DE LIMPIEZA DEL EQUIPO ---
    # Paso 1: Base (Borrado de nombres, estandarización de strings)
    df_temp = base_cleaning.clean_base_columns(df_temp)
    
    # Paso 2: Codificadores pesados
    df_temp = city_bank.get_city_bank_encoder(df_temp)
    
    # Paso 3: Variables de preprocesamiento estático (Sin opciones)
    df_temp = noemp.preprocess_noemp(df_temp, option="log")
    df_temp = franchise_code.preprocess_franchise_code(df_temp, option="binary", source_col="FranchiseCode")
    df_temp = urban_rural.preprocess_urban_rural(df_temp, option="onehot", source_col="UrbanRural")
    df_temp = RevLineCr.preprocess_revlinecr(df_temp, option="C", source_col="RevLineCr")
    df_temp = LowDoc.preprocess_lowdoc(df_temp, option="C", source_col="LowDoc")
    df_temp = disbursementDate.preprocess_disbursementdate(df_temp, source_col="DisbursementDate")
    df_temp = balanceGross.preprocess_balancegross(df_temp, source_col="BalanceGross")
    df_temp = accept.preprocess_accept(df_temp, source_col="Accept")
    
    # Paso 4: Variables Dinámicas (Asignadas por el torneo)
    df_temp = newExists.preprocess_newexist(df_temp, option=combo[0], source_col="NewExist")
    df_temp = createJob.preprocess_createjob(df_temp, option=combo[1], source_col="CreateJob")
    df_temp = retainedJob.preprocess_retainedjob(df_temp, option=combo[2], source_col="RetainedJob")
    df_temp = approvalDate.preprocess_approvaldate(df_temp, option=combo[3], source_col="ApprovalDate")
    df_temp = approvalFY.preprocess_approvalfy(df_temp, option=combo[4], source_col="ApprovalFY")
    df_temp = disbursementGross.preprocess_disbursementgross(df_temp, option=combo[5], source_col="DisbursementGross")
    
    # 🚨 FILTRO DE SEGURIDAD 🚨
    # Conservamos exclusivamente columnas numéricas para evitar errores con fechas sobrantes (datetime64)
    df_temp = df_temp.select_dtypes(include=['int16', 'int32', 'int64', 'float16', 'float32', 'float64', 'bool', 'uint8', 'Int64', 'Float64'])
    
    # Separación de datos
    X = df_temp.drop(columns=['Accept'])
    y = df_temp['Accept']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Entrenamiento rápido
    modelo_temp = xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
    modelo_temp.fit(X_train, y_train)
    y_pred = modelo_temp.predict(X_test)
    
    f1 = f1_score(y_test, y_pred, average='macro')
    
    # Evaluamos y guardamos al ganador
    if f1 > mejor_f1:
        mejor_f1 = f1
        mejor_combo = combo
        mejor_df = df_temp

# ==========================================
# 4. RESULTADOS DEL TORNEO
# ==========================================
print(f"\n🏆 ¡Torneo finalizado! La combinación ganadora fue: {dict(zip(keys, mejor_combo))}")
print(f"📊 F1-Score Base obtenido: {mejor_f1:.4f}")
print(f"🧩 Columnas finales utilizadas: {mejor_df.shape[1]}")

⏳ Cargando entorno de Feature Engineering...
===Loaded preprocessing modules: LowDoc, RevLineCr, accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, newExists, noemp, retainedJob, urban_rural
Current df features:
1. City_0
2. City_1
3. City_2
4. City_3
5. City_4
6. City_5
7. City_6
8. City_7
9. City_8
10. City_9
11. City_10
12. Bank_0
13. Bank_1
14. Bank_2
15. Bank_3
16. Bank_4
17. Bank_5
18. Bank_6
19. Bank_7
20. Bank_8
21. ApprovalDate
22. ApprovalFY
23. NewExist
24. CreateJob
25. RetainedJob
26. FranchiseCode
27. UrbanRural
28. RevLineCr
29. LowDoc
30. DisbursementDate
31. DisbursementGross
32. BalanceGross
33. Accept
34. IsLocalBank
35. NoEmp_Log
NewExist option used: A
Rows before: 20,768
Rows after: 20,768
CreateJob option used: A
Rows before: 20,768
Rows after: 20,768
RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768
ApprovalDate option used: A
Rows before: 20,768
Rows 

,approvalyear_normalized,approvalmonth_normalized
0,0.590909,0.636364
1,0.840909,1.000000
2,0.590909,0.363636
3,0.909091,0.909091
4,0.840909,0.363636
5,0.750000,0.181818
6,0.795455,0.818182
7,0.522727,0.090909
8,0.659091,0.363636
9,0.772727,0.181818


ApprovalFY option used: A
Rows before: 20,768
Rows after: 20,768

Opción A seleccionada: La columna 'ApprovalFY' fue ELIMINADA del dataset.


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,approvalyear_normalized,approvalmonth_normalized
0,0,0,0,0,0,0,0,0,0,0,...,$0.00,0,1,3.367296,0,0,0.0,0.0,0.590909,0.636364
1,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,0.693147,1,0,0.000114,0.000114,0.840909,1.000000
2,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,1.945910,1,0,0.0,0.0,0.590909,0.363636


Opción de FranchiseCode utilizada: binary
Filas antes: 20,768
Filas después: 20,768


...

Rows before: 20,768
RevLineCr option used: C
Rows after: 20,768
Rows before: 20,768
LowDoc option used: C
Rows before: 20,768
Rows after: 20,768
DisbursementGross option used: B
Rows before: 20,768
Rows after: 20,768


,DisbursementGross
0,1.469113
1,-0.572057
2,-0.59124
3,-0.395861
4,-0.48467
5,-0.243111
6,-0.48467
7,-0.129436
8,2.890045
9,-0.542573


Total features: 40


,Feature,Type
0,City_0,int64
1,City_1,int64
2,City_2,int64
3,City_3,int64
4,City_4,int64
5,City_5,int64
6,City_6,int64
7,City_7,int64
8,City_8,int64
9,City_9,int64


⏳ Iniciando torneo de 64 combinaciones A/B (esto tomará unos minutos)...

🏆 ¡Torneo finalizado! La combinación ganadora fue: {'newExists': 'A', 'createJob': 'A', 'retainedJob': 'A', 'approvalDate': 'A', 'approvalFY': 'B', 'disbursementGross': 'A'}
📊 F1-Score Base obtenido: 0.7097
🧩 Columnas finales utilizadas: 41


## Fase 1: Entrenamiento Profundo y Hyperparameter Tuning
Ya tenemos en memoria el conjunto de datos ganador con la mejor combinación de preprocesamiento. Ahora utilizaremos **GridSearchCV** para encontrar la mejor arquitectura matemática para nuestro XGBoost. Optimizaremos la cantidad de árboles (`n_estimators`), la profundidad máxima (`max_depth`) y el submuestreo (`subsample`) para garantizar que el modelo no memorice (*overfitting*).

In [2]:
print("⏳ Buscando los mejores hiperparámetros para la combinación ganadora (CV=3)...")

# 1. Preparamos los datos del dataset ganador
X_best = mejor_df.drop(columns=['Accept'])
y_best = mejor_df['Accept']

# Hacemos el split final para el modelo afinado
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_best, y_best, test_size=0.2, random_state=42, stratify=y_best
)

# 2. Configuración del modelo base
xgb_base = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# Grilla de hiperparámetros
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# 3. Búsqueda exhaustiva
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

# Entrenamiento
grid_search.fit(X_train_b, y_train_b)
best_xgb = grid_search.best_estimator_

print("\n✅ Tuning terminado.")
print("🏆 Mejores hiperparámetros encontrados para XGBoost:")
print(grid_search.best_params_)
print("-" * 50)

⏳ Buscando los mejores hiperparámetros para la combinación ganadora (CV=3)...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

✅ Tuning terminado.
🏆 Mejores hiperparámetros encontrados para XGBoost:
{'colsample_bytree': 0.8, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
--------------------------------------------------


## Fase 2: Evaluación Final
Comparamos las predicciones de nuestro modelo afinado (`best_xgb`) contra los datos de prueba (`X_test_b`). Empleamos una función de reporte estandarizada para alinearnos con las métricas requeridas por el equipo (Accuracy, Precision, Recall, F1-Score y ROC-AUC).

In [3]:
# Definimos la función de reporte estándar del equipo
def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    roc_auc = roc_auc_score(y_test, y_prob)
    
    print(f"\n=== {name} ===")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Precision (Macro): {prec:.4f}")
    print(f"Recall (Macro)   : {rec:.4f}")
    print(f"F1-score (Macro) : {f1:.4f}")
    print(f"ROC-AUC          : {roc_auc:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "ROC-AUC": roc_auc}

# Ejecutamos el reporte final con nuestro modelo ganador
xgb_results = evaluate_model(best_xgb, X_test_b, y_test_b, name="XGBoost Tuned (Winner Combo)")


=== XGBoost Tuned (Winner Combo) ===
Accuracy         : 0.8175
Precision (Macro): 0.7496
Recall (Macro)   : 0.6825
F1-score (Macro) : 0.7041
ROC-AUC          : 0.8148

Classification Report:
              precision    recall  f1-score   support

         0.0       0.65      0.43      0.52       950
         1.0       0.85      0.93      0.89      3204

    accuracy                           0.82      4154
   macro avg       0.75      0.68      0.70      4154
weighted avg       0.80      0.82      0.80      4154



## Conclusión Individual: Modelo XGBoost

**¿Qué se hizo?**
Para evitar sesgos humanos en la elección de la base de datos, implementamos un "Torneo" automatizado que evaluó el rendimiento del algoritmo XGBoost frente a las 64 combinaciones posibles de *Feature Engineering* (Opciones A y B creadas por el equipo). Una vez que el algoritmo seleccionó matemáticamente su base de datos ideal, aplicamos una búsqueda exhaustiva (`GridSearchCV`) para afinar su arquitectura interna y prevenir que el modelo memorizara los datos (*overfitting*).

**Resultados Detallados:**
* **La Base de Datos Ideal:** El algoritmo determinó empíricamente que rinde mejor cuando se le entregan los datos normalizados (Opción A en `newExists`, `createJob`, `retainedJob`, `approvalDate` y `disbursementGross`), con la única excepción del Año Fiscal (`approvalFY`), donde prefirió la Opción B.
* **Métricas Alcanzadas:** Con su configuración óptima y conservadora (`max_depth=3`, `n_estimators=200`), el modelo logró un **Accuracy general del 81.75%** y un **F1-Score Macro de 0.7041**.
* **Área de Oportunidad (El desbalance):** Aunque el modelo es extraordinariamente preciso para identificar préstamos aprobados (clase 1.0, con un 93% de acierto), tiene dificultades para identificar los préstamos rechazados (clase 0.0), logrando capturar solo el 43% de estos casos. Esto refleja que XGBoost requiere un tratamiento especial y manual para manejar bases de datos desbalanceadas.

## RANDOM FOREST

### 1. IMPORTS + CONFIGURACIÓN

In [1]:
# === Imports generales ===
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import importlib

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

### 2. CARGAR PROYECTO + PREPROCESSING

In [2]:
# === Configurar paths ===
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# === Auto-import preprocessing modules ===
from importlib import import_module

preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"

for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("Loaded preprocessing:", ", ".join(preprocessing_modules.keys()))

Loaded preprocessing: accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, LowDoc, newExists, noemp, retainedJob, RevLineCr, urban_rural


### 3. CARGAR DATASET

In [3]:
# === Load dataset ===
df = pd.read_csv(project_root / "data" / "train.csv")

print("Shape:", df.shape)
df.head()

Shape: (20768, 21)


,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


### 4. APLICAR TODO EL PREPROCESSING

In [4]:
# === Base cleaning ===
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)

# === NoEmp ===
noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option="log")

# === City + Bank ===
city_bank = importlib.reload(city_bank)
df = city_bank.get_city_bank_encoder(df)

# === NewExist ===
newExists = importlib.reload(newExists)
df = newExists.preprocess_newexist(df, option="B", source_col="NewExist")

# === CreateJob ===
createJob = importlib.reload(createJob)
df = createJob.preprocess_createjob(df, option="A", source_col="CreateJob")

# === RetainedJob ===
retainedJob = importlib.reload(retainedJob)
df = retainedJob.preprocess_retainedjob(df, option="B", source_col="RetainedJob")

# === ApprovalDate ===
approvalDate = importlib.reload(approvalDate)
df = approvalDate.preprocess_approvaldate(df, option="A", source_col="ApprovalDate")

# === ApprovalFY ===
approvalFY = importlib.reload(approvalFY)
df = approvalFY.preprocess_approvalfy(df, option="B", source_col="ApprovalFY")

# === Franchise ===
franchise_code = importlib.reload(franchise_code)
df = franchise_code.preprocess_franchise_code(df, option="binary", source_col="FranchiseCode")

# === UrbanRural ===
urban_rural = importlib.reload(urban_rural)
df = urban_rural.preprocess_urban_rural(df, option="onehot", source_col="UrbanRural")

# === RevLineCr ===
RevLineCr = importlib.reload(RevLineCr)
df = RevLineCr.preprocess_revlinecr(df, option="C", source_col="RevLineCr")

# === LowDoc ===
LowDoc = importlib.reload(LowDoc)
df = LowDoc.preprocess_lowdoc(df, option="C", source_col="LowDoc")

# === DisbursementDate ===
disbursementDate = importlib.reload(disbursementDate)
df = disbursementDate.preprocess_disbursementdate(df, source_col="DisbursementDate")

# === BalanceGross ===
balanceGross = importlib.reload(balanceGross)
df = balanceGross.preprocess_balancegross(df, source_col="BalanceGross")

# === Accept ===
accept = importlib.reload(accept)
df = accept.preprocess_accept(df, source_col="Accept")

# === DisbursementGross ===
disbursementGross = importlib.reload(disbursementGross)
df = disbursementGross.preprocess_disbursementgross(df, option="B", source_col="DisbursementGross")

print("Final shape:", df.shape)

Final shape: (20765, 40)


### 5. SPLIT (COMÚN PARA TODOS)

In [5]:
# === Split ===
X = df.drop(columns=["Accept"])
y = df["Accept"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (16612, 39)
Test shape: (4153, 39)


### 6. FUNCIÓN DE EVALUACIÓN MACRO 

In [6]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred,average= "macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average= "macro", zero_division=0)
    roc = roc_auc_score(y_test, y_prob)

    print(f"\n=== {name} ===")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1-score (MACRO) :", round(f1, 4))
    print("ROC-AUC  :", round(roc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_macro": f1,
        "roc_auc": roc
    }

### RANDOM FOREST BASE

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

evaluate_model(rf_model, X_test, y_test, name="Random Forest Base")


=== Random Forest Base ===
Accuracy : 0.8225
Precision: 0.7628
Recall   : 0.6828
F1-score (MACRO) : 0.707
ROC-AUC  : 0.8231

Classification Report:
              precision    recall  f1-score   support

         0.0       0.68      0.43      0.52       950
         1.0       0.85      0.94      0.89      3203

    accuracy                           0.82      4153
   macro avg       0.76      0.68      0.71      4153
weighted avg       0.81      0.82      0.81      4153



{'accuracy': 0.8225379243920058,
 'precision': 0.7627672991624981,
 'recall': 0.6828157812577025,
 'f1_macro': 0.7069849015077432,
 'roc_auc': 0.8231444205268088}

### RANDOM FOREST HIPERPARÁMETROS (TUNED)

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Modelo base para tuning
rf = RandomForestClassifier(random_state=42)

# Grid de hiperparámetros
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"]
}

# GridSearch
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

# Entrenamiento
grid_search.fit(X_train, y_train)

# Mejor modelo
best_rf = grid_search.best_estimator_

print("Mejores parámetros:")
print(grid_search.best_params_)

# Evaluación
tuned_results = evaluate_model(best_rf, X_test, y_test, name="Random Forest Tuned")

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Mejores parámetros:
{'class_weight': 'balanced', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}

=== Random Forest Tuned ===
Accuracy : 0.8086
Precision: 0.7286
Recall   : 0.7274
F1-score (MACRO) : 0.728
ROC-AUC  : 0.8249

Classification Report:
              precision    recall  f1-score   support

         0.0       0.58      0.58      0.58       950
         1.0       0.88      0.88      0.88      3203

    accuracy                           0.81      4153
   macro avg       0.73      0.73      0.73      4153
weighted avg       0.81      0.81      0.81      4153



## Conclusión Global del "Tree Team": XGBoost vs. Random Forest

**¿Qué se hizo a nivel de equipo?**
Se sometieron a prueba dos de los algoritmos basados en árboles más potentes de la industria (Random Forest y XGBoost), utilizando las variables limpias y codificadas generadas en la etapa de preprocesamiento. Esto garantizó que ambos algoritmos compitieran en igualdad de condiciones y bajo altos estándares de exigencia técnica.

**El Veredicto y los Resultados:**
Tras evaluar los modelos afinados, **el modelo Random Forest se declara como el ganador definitivo del equipo**, superando a XGBoost por razones clave enfocadas en el negocio:

* **Mayor Equilibrio Global (F1-Score):** Random Forest alcanzó un **F1-Score Macro de 0.728**, superando el 0.7041 de XGBoost. Esto indica que es un modelo matemáticamente más estable en términos generales.
* **Solución a la Clase Minoritaria (El gran diferenciador):** El modelo Random Forest incluyó el parámetro de balanceo automático (`class_weight='balanced'`). Gracias a esto, logró identificar el **58% de los préstamos rechazados (clase 0.0)**, superando por un margen de 15 puntos porcentuales a XGBoost (que solo identificó el 43%). En el mundo real, detectar a tiempo un préstamo con riesgo de fracaso es una de las métricas de mayor valor para una institución financiera.
* **Estabilidad y Transparencia:** Ambos modelos confirmaron que aprovechar el 100% de las variables (incluyendo ciudades y bancos codificados de forma binaria) aporta valor real sin colapsar la memoria computacional. Además, al ser modelos basados en árboles, permiten extraer la importancia de cada variable para ofrecer explicaciones claras y transparentes al área de negocio.

**Recomendación Final:**
El equipo debe avanzar utilizando el **Random Forest Afinado** como su modelo principal para la clasificación de los préstamos. Como trabajo futuro, se recomienda explorar un modelo "Ensamblado" (*Voting Classifier*) que combine la excelente tasa de detección de préstamos aprobados de XGBoost con la gran sensibilidad para detectar préstamos rechazados de Random Forest.